# Day 18 — PCA (Principal Component Analysis) 🧠

A practical PCA project using the Breast Cancer Wisconsin dataset. The notebook compares Logistic Regression before and after dimensionality reduction.

## 🎯 Objectives

- Understand dimensionality reduction
- Learn Principal Component Analysis (PCA)
- Standardize features before PCA
- Analyze explained variance
- Select components retaining at least 95% variance
- Visualize data in 2D
- Compare model performance before and after PCA

## 📊 Dataset

The Scikit-learn Breast Cancer Wisconsin (Diagnostic) dataset contains 569 samples and 30 numerical features for binary classification.

## 🛠️ Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## 📥 Load Dataset

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print('Dataset Shape:', X.shape)
print('Number of Features:', X.shape[1])
display(X.head())

## 🔎 Explore Dataset

In [ ]:
print('Target Distribution:')
print(y.value_counts())
print('\nTarget Names:', data.target_names)
print('\nMissing Values:', X.isnull().sum().sum())

## ✂️ Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Training Samples:', len(X_train))
print('Testing Samples :', len(X_test))

## 📐 Feature Scaling

PCA is sensitive to feature magnitudes, so features are standardized first. The scaler is fitted only on training data to avoid data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Feature scaling completed.')

## 🤖 Baseline Model — Without PCA

In [ ]:
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_scaled, y_train)
baseline_pred = baseline_model.predict(X_test_scaled)
baseline_accuracy = accuracy_score(y_test, baseline_pred)
print(f'Baseline Accuracy: {baseline_accuracy:.4f}')

## 🧠 What is PCA?

PCA transforms the original features into new variables called principal components. The first components capture the largest amounts of variance in the data. This allows high-dimensional data to be represented using fewer dimensions.

## 🔬 Apply PCA — All Components

In [ ]:
pca_full = PCA()
X_train_pca_full = pca_full.fit_transform(X_train_scaled)
X_test_pca_full = pca_full.transform(X_test_scaled)

print('Original Features:', X_train_scaled.shape[1])
print('PCA Components   :', X_train_pca_full.shape[1])

## 📊 Explained Variance Ratio

In [ ]:
explained_variance = pca_full.explained_variance_ratio_
variance_table = pd.DataFrame({
    'Component': [f'PC{i}' for i in range(1, len(explained_variance)+1)],
    'Explained Variance': explained_variance
})
display(variance_table.head(10))

## 📈 Cumulative Explained Variance

In [ ]:
cumulative_variance = explained_variance.cumsum()
cumulative_table = pd.DataFrame({
    'Number of Components': range(1, len(cumulative_variance)+1),
    'Cumulative Variance': cumulative_variance
})
display(cumulative_table.head(15))

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, marker='o')
plt.axhline(y=0.90, linestyle='--', label='90% Variance')
plt.axhline(y=0.95, linestyle='--', label='95% Variance')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — Cumulative Explained Variance')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 🎯 Select Components for 95% Variance

In [ ]:
n_components_95 = (cumulative_variance >= 0.95).argmax() + 1
print('Components required for 95% variance:', n_components_95)

## 🔄 Transform Dataset Using Selected Components

In [ ]:
pca = PCA(n_components=n_components_95)
X_train_reduced = pca.fit_transform(X_train_scaled)
X_test_reduced = pca.transform(X_test_scaled)

print('Original Feature Count:', X_train_scaled.shape[1])
print('Reduced Feature Count :', X_train_reduced.shape[1])
print('Variance Retained      :', pca.explained_variance_ratio_.sum())

## 🤖 Logistic Regression After PCA

In [ ]:
pca_model = LogisticRegression(max_iter=1000)
pca_model.fit(X_train_reduced, y_train)
pca_pred = pca_model.predict(X_test_reduced)
pca_accuracy = accuracy_score(y_test, pca_pred)
print(f'PCA Model Accuracy: {pca_accuracy:.4f}')

## 🔬 Compare Model Performance

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression - Without PCA', 'Logistic Regression - With PCA'],
    'Accuracy': [baseline_accuracy, pca_accuracy]
})
display(comparison)
print(f'Accuracy Difference: {pca_accuracy - baseline_accuracy:.4f}')

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(comparison['Model'], comparison['Accuracy'])
plt.ylim(0,1)
plt.ylabel('Accuracy')
plt.title('Logistic Regression — Before vs After PCA')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 🗺️ 2D PCA Visualization

In [ ]:
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_train_scaled)

plt.figure(figsize=(9,6))
plt.scatter(X_2d[y_train == 0, 0], X_2d[y_train == 0, 1], label=data.target_names[0], alpha=0.7)
plt.scatter(X_2d[y_train == 1, 0], X_2d[y_train == 1, 1], label=data.target_names[1], alpha=0.7)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('2D PCA Visualization')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 📋 PCA Component Information

In [ ]:
pca_information = pd.DataFrame({
    'Component': [f'PC{i}' for i in range(1, len(pca.explained_variance_ratio_)+1)],
    'Explained Variance': pca.explained_variance_ratio_,
    'Cumulative Variance': pca.explained_variance_ratio_.cumsum()
})
display(pca_information.head(10))

## 🔲 Confusion Matrix — PCA Model

In [ ]:
cm = confusion_matrix(y_test, pca_pred)
print(cm)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=data.target_names)
disp.plot()
plt.title('PCA Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

## 📋 Classification Report

In [ ]:
print(classification_report(y_test, pca_pred, target_names=data.target_names))

## 📉 Feature Reduction Summary

In [ ]:
original_features = X_train_scaled.shape[1]
reduced_features = X_train_reduced.shape[1]
reduction_percentage = (1 - reduced_features / original_features) * 100
variance_retained = pca.explained_variance_ratio_.sum()

print('Original Features :', original_features)
print('Reduced Features  :', reduced_features)
print(f'Feature Reduction : {reduction_percentage:.2f}%')
print(f'Variance Retained : {variance_retained:.4f}')

## 🔑 Key Findings

- PCA reduces the dimensionality of the dataset.
- Standardization is performed before PCA.
- Explained variance helps determine how many components to retain.
- Components retaining at least 95% variance are selected.
- Logistic Regression performance is compared before and after PCA.
- Two principal components can be used to visualize high-dimensional data.

## 🧠 Key Learnings

- Dimensionality reduction
- Principal components
- Explained variance
- Cumulative explained variance
- Feature scaling before PCA
- Selecting the number of components
- 2D PCA visualization
- Comparing model performance

## 🏁 Conclusion

PCA is a useful dimensionality reduction technique for representing high-dimensional data with fewer components. In this project, the dataset was standardized and PCA was used to retain at least 95% of the variance. Logistic Regression was then trained on the reduced representation and compared with a baseline model.

## 📚 References

- Scikit-learn — PCA: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
- Scikit-learn — StandardScaler: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
- Scikit-learn — Logistic Regression: https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
- Scikit-learn — Breast Cancer Dataset: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html

## 📅 30 Days of Machine Learning

**Day 18/30 — PCA 🧠**

Learning → Coding → Experimenting → Reducing Dimensions → Evaluating 🚀